# NB_hospital_daily_summary
## Silver → Gold |  Health Network

**Target Table:** `adf_healthgroup_metadatadriven.gold.hospital_daily_summary`  
**Grain:** One row per hospital per calendar day  
**Load Type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT`  
**Source:** STTM `sttm_silver_to_gold.xlsx`, Sheet `hospital_daily_summary`

### Source Tables
| Table | Role |
|---|---|
| `adf_healthgroup_metadatadriven.silver.lab_results` | Driving table; grouped by `hospital_id`, `test_date` |
| `adf_healthgroup_metadatadriven.silver.hospitals` | LEFT JOIN on `hospital_id` |
| `adf_healthgroup_metadatadriven.silver.calendar` | Optional LEFT JOIN on `date_key` (INT) = `CAST(test_date AS INT string via date_format)` |

### Notes
- Calendar join key: `calendar.date_key` is stored as `INT` formatted `yyyyMMdd`. Join condition: `c.date_key = CAST(date_format(lr.test_date, 'yyyyMMdd') AS INT)`.
- `abnormal_rate_pct` and `critical_rate_pct` are `NULL` when `total_tests = 0` (NULLIF guard).
- `test_volume_change_pct` is `NULL` for a hospital’s first observation day.
- Window function: `LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)` for day-over-day comparison.

In [0]:
# Set the Unity Catalog catalog name used throughout this notebook
catalog = "adf_healthgroup_metadatadriven"
print(f"Using catalog: {catalog}")

In [0]:
%sql
-- ============================================================
-- NB_hospital_daily_summary
-- Target: adf_healthgroup_metadatadriven.gold.hospital_daily_summary
-- Grain : One row per hospital per calendar day
-- Load  : Full rebuild (CREATE OR REPLACE)
-- ============================================================

CREATE OR REPLACE TABLE adf_healthgroup_metadatadriven.gold.hospital_daily_summary AS

WITH daily_base AS (
    -- Aggregate lab_results to hospital + day grain
    SELECT
        lr.hospital_id,
        lr.test_date                                                         AS summary_date,
        COUNT(*)                                                             AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                        AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                         AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)            AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)            AS critical_test_count
    FROM adf_healthgroup_metadatadriven.silver.lab_results AS lr
    GROUP BY lr.hospital_id, lr.test_date
),

daily_with_rates AS (
    -- Add derived percentage columns and LAG-based prior day
    SELECT
        db.*,
        ROUND(
            db.abnormal_test_count / NULLIF(db.total_tests, 0) * 100.0, 2
        )                                                                    AS abnormal_rate_pct,
        ROUND(
            db.critical_test_count / NULLIF(db.total_tests, 0) * 100.0, 2
        )                                                                    AS critical_rate_pct,
        LAG(db.total_tests) OVER (
            PARTITION BY db.hospital_id
            ORDER BY db.summary_date
        )                                                                    AS prev_day_total_tests
    FROM daily_base AS db
)

SELECT
    dwr.hospital_id,
    h.hospital_name,
    dwr.summary_date,

    -- Optional calendar attributes
    c.day_name,
    c.is_weekend,

    -- Metrics
    dwr.total_tests,
    dwr.unique_patients_tested,
    dwr.active_doctor_count,
    dwr.abnormal_test_count,
    dwr.critical_test_count,
    dwr.abnormal_rate_pct,
    dwr.critical_rate_pct,

    -- Day-over-day trend
    dwr.prev_day_total_tests,
    ROUND(
        (dwr.total_tests - dwr.prev_day_total_tests)
        / NULLIF(dwr.prev_day_total_tests, 0) * 100.0, 2
    )                                                                        AS test_volume_change_pct

FROM daily_with_rates                                           AS dwr
LEFT JOIN adf_healthgroup_metadatadriven.silver.hospitals       AS h
    ON dwr.hospital_id = h.hospital_id
LEFT JOIN adf_healthgroup_metadatadriven.silver.calendar        AS c
    ON c.date_key = CAST(date_format(dwr.summary_date, 'yyyyMMdd') AS INT)

ORDER BY dwr.hospital_id, dwr.summary_date

In [0]:
%sql
-- ============================================================
-- Post-load validation
-- ============================================================

SELECT
    COUNT(*)                                                         AS total_rows,
    COUNT(DISTINCT hospital_id)                                      AS distinct_hospitals,
    MIN(summary_date)                                                AS earliest_date,
    MAX(summary_date)                                                AS latest_date,
    SUM(CASE WHEN hospital_id  IS NULL THEN 1 END)                   AS null_hospital_id,
    SUM(CASE WHEN summary_date IS NULL THEN 1 END)                   AS null_summary_date,
    SUM(CASE WHEN total_tests  <= 0    THEN 1 END)                   AS zero_or_negative_tests,
    SUM(CASE WHEN abnormal_rate_pct > 100 THEN 1 END)                AS rate_over_100_pct
FROM adf_healthgroup_metadatadriven.gold.hospital_daily_summary

In [0]:
updated_df = spark.table("adf_healthgroup_metadatadriven.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))